# Training a Forecasting Model with Buildyn

This notebook demonstrates a practical use case for the **Buildyn** library: generating synthetic building simulation data and using it to train a neural network for time-series forecasting.

We will:
- Configure a Functional Mock-up Unit (FMU) representing a building system
- Use Buildyn to generate realistic simulation data
- Train an LSTM neural network on the generated data
- Evaluate the training performance


## 1. Install Dependencies

If you are running this notebook in a fresh environment, you neeed to have the mo_prior package and its dependencies installed first.

The packages installed here are only used fo the example use case and are not needed to generate datasets.

In [ ]:
# Install missing dependencies (run once) for gpu environment install torch with cuda support
!pip install scikit-learn 

!pip install torch 
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

## 2. Import Libraries

Here we import Buildyn components, the configured FMU helper, and PyTorch utilities for building and training our neural network.

In [ ]:
from builda_fmu import get_configured_builda_fmu
from buildyn.distributions.continuous.gauss_distribution import GaussDistribution
from buildyn.buildyn import BuilDyn

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

## 3. Configure the FMU and Buildyn Prior

As shown in the previus notebooks we configure an FMU and simulate some data for the validation set

In [ ]:
# Configure FMU and prior
fmu = get_configured_builda_fmu(internal_controller=True)

observables = ["thermalZone.TAir", "ctrSignalHeating"]
prior = BuilDyn(fmu, observables=observables)

g_dist = GaussDistribution(mu=0.6, sigma=0.2, min=0.1, max=1.2)
prior.add_variation_distribution("UExt", g_dist)

# Validation dataset
val_set = prior.sample_one(stop_time=900424*180)

## 4. Define the LSTM Model

We implement a Long Short-Term Memory (LSTM) network, a popular architecture for time-series forecasting. In this example, the model learns to predict future building states from previously observed signals.

In [ ]:
class LSTM(nn.Module):
    def __init__(
        self,
        num_features: int = 8,
        hidden_size: int = 8,
        num_layers: int = 1,
        forecast_horizon: int = 1,
        dropout: float = 0.0,
        num_targets: int = 1,
    ):
        super().__init__()
        self.forecast_horizon = forecast_horizon
        self.num_targets = num_targets

        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.fc = nn.Linear(hidden_size, forecast_horizon * num_targets)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]
        lstm_out = self.fc(lstm_out)
        return lstm_out.view(-1, self.forecast_horizon, self.num_targets)

## 5. Data Preprocessing

Before training, the simulated data must be converted into sequences suitable for the LSTM. This step includes:
- Selecting features and targets
- Scaling values
- Creating sliding windows
- Packaging the data into batches

In [ ]:
def preprocess_lstm_data(
    data,
    feature_cols,
    target_cols,
    seq_len=96,
    forecast_horizon=1,
    batch_size=16,
    scaler=None,
    shuffle=True,
):
    df = pd.DataFrame(data)

    if scaler is None:
        scaler = MinMaxScaler()
        scaler.fit(df[feature_cols])

    scaled = scaler.transform(df[feature_cols]).astype(np.float32)

    num_samples = len(scaled) - seq_len - forecast_horizon + 1
    target_idx = [feature_cols.index(c) for c in target_cols]

    X = np.zeros((num_samples, seq_len, len(feature_cols)), dtype=np.float32)
    Y = np.zeros((num_samples, forecast_horizon, len(target_cols)), dtype=np.float32)

    for i in range(num_samples):
        X[i] = scaled[i:i+seq_len]
        Y[i] = scaled[i+seq_len:i+seq_len+forecast_horizon, target_idx]

    dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(Y))

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,  # safer default for notebooks
    )

    return loader, scaler

## 6. Model Setup

We configure the training device (GPU if available), instantiate the model, and prepare the optimizer and loss function.

In [ ]:
# Model setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

model = LSTM(num_features=2, hidden_size=16, num_layers=2, forecast_horizon=5, num_targets=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Validation loader
val_loader, scaler = preprocess_lstm_data(
    data=val_set,
    feature_cols=['thermalZone.TAir', 'ctrSignalHeating'],
    target_cols=['thermalZone.TAir'],
    seq_len=96,
    forecast_horizon=5,
    batch_size=128,
    scaler=None,
    shuffle=False,
)

## 7. Training with Synthetic Data

Instead of relying on a fixed dataset, we repeatedly sample fresh trajectories from the Buildyn prior at every epoch.

In [ ]:
print('Starting training...')
train_curve, val_curve = [], []

for epoch in range(10):
    print('generating new training data...')
    train_set = prior.sample_one(stop_time=900424*180)

    print('preprocessing training data...')
    train_loader, scaler = preprocess_lstm_data(
        data=train_set,
        feature_cols=['thermalZone.TAir', 'ctrSignalHeating'],
        target_cols=['thermalZone.TAir'],
        seq_len=96,
        forecast_horizon=5,
        batch_size=128,
        scaler=scaler,
        shuffle=True,
    )

    model.train()
    train_losses = []
    print('training epoch...')
    for batch_X, batch_Y in train_loader:
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
            outputs = model(batch_X)
            val_losses.append(criterion(outputs, batch_Y).item())

    train_loss = sum(train_losses)/len(train_losses)
    val_loss = sum(val_losses)/len(val_losses)

    train_curve.append(train_loss)
    val_curve.append(val_loss)

    print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

## 8. Visualize Training Performance

Finally, we plot the training and validation loss curves to understand how well the model is learning.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_curve, label='Training Loss', marker='o')
plt.plot(val_curve, label='Validation Loss', marker='s')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()